# Sentinel-2 Median Composite Downloader

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/S2_Median.ipynb)

This notebook downloads **cloud-masked Sentinel-2 Surface Reflectance median composites**
for a chosen Area of Interest (AOI), split into user-defined multi-month periods, for a range of years.

It uses the **Google Earth Engine (GEE) Python API** and **geemap** to:
1. Filter the `COPERNICUS/S2_SR_HARMONIZED` collection by date, AOI, and cloud percentage.
2. Mask clouds/cirrus using the `QA60` band.
3. Reduce each period's image collection to a per-pixel **median**.
4. Export the resulting bands (10 m and 20 m) as GeoTIFFs, organized by year/period.
5. Save per-image metadata (index + cloud cover %) to a CSV file.

> Replace `YOUR_USERNAME/YOUR_REPO` above with your actual GitHub path once this file is pushed,
> so the "Open in Colab" badge works.


## 1. Setup

If you are running this in **Google Colab**, `geemap` and the Earth Engine API need to be
installed first (Colab does not have them pre-installed). Uncomment and run the cell below
on first use. If running locally with an environment that already has these packages, you can skip it.


In [ ]:
# Uncomment the line below the first time you run this in Google Colab
# (or in any fresh environment that doesn't already have these packages installed)
# !pip install --upgrade earthengine-api geemap

In [ ]:
# Core imports
import os
import pandas as pd
import ee
import geemap

## 2. Authenticate & Initialize Earth Engine

`ee.Authenticate()` opens a browser/Colab prompt to sign in with your Google account and
generate a credentials token (only needed once per machine/session).
`ee.Initialize()` starts the Earth Engine session for the rest of the notebook.

> You need a Google Earth Engine account (free, but requires registration at
> https://code.earthengine.google.com/register) before this will work.


In [ ]:
ee.Authenticate()
ee.Initialize()  # add project='your-gee-project-id' if your account requires a cloud project

## 3. Interactive Map (optional)

Use this map to visually navigate to your area of interest and draw a rectangle/polygon
with the drawing tools, if you don't already have AOI coordinates.


In [ ]:
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.setCenter(17.926, 47.745, 12)  # (longitude, latitude, zoom) -- change to your region of interest

## 4. Define your Area of Interest (AOI)

**Keep your AOI under ~250 km²** to keep exports fast and within Earth Engine's export limits.

There are two ways to set the `roi` variable used later in the notebook:

**Option A — Draw it on the map above:**
1. Run the `Map` cell above.
2. Use the rectangle/polygon drawing tool on the map to draw your AOI.
3. Run the cell below to capture it as `roi`.

**Option B — Paste coordinates directly:**
If you already have a geometry (e.g. exported earlier with `.getInfo()`), paste the
coordinate list into the `ee.Geometry.Polygon(...)` cell further down instead.


In [ ]:
# Option A: capture the AOI you drew on the interactive map above
# if Map.user_roi is not None:
#     roi = Map.user_roi
#     print(roi.getInfo())

In [ ]:
# Option B: define the AOI manually by pasting a geometry coordinate list
# roi = ee.Geometry.Polygon(
#     [Paste the Geometry coordinate list here]
# )

# NOTE: `roi` MUST be defined (via Option A or B) before running the export cell below.

## 5. Select the Range of Years

Set the first and last year (inclusive) for which you want to download composites.


In [ ]:
years = list(range(2017, 2025))  # e.g. 2017-2024 inclusive

years

## 6. Sentinel-2 Cloud Masking & Band Setup

In [ ]:
def maskS2clouds(image):
    """
    Mask clouds and cirrus in a Sentinel-2 SR image using the QA60 band,
    and rescale reflectance values from [0, 10000] to [0, 1].

    Bits 10 and 11 of QA60 flag clouds and cirrus respectively;
    a pixel is kept only if BOTH bits are 0 (clear sky).
    """
    qa = image.select('QA60')

    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
        .And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    return image.updateMask(mask).divide(10000)

In [ ]:
# Sentinel-2 band groups by native spatial resolution
band_or10 = ['B2', 'B3', 'B4', 'B8']                       # 10 m bands (Blue, Green, Red, NIR)
band_or20 = ['B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'QA60']  # 20 m bands (Red Edge, SWIR, cloud mask)

# After ee.Reducer.median(), Earth Engine appends '_median' to each band name
band_10 = [band + '_median' for band in band_or10]
band_20 = [band + '_median' for band in band_or20]

# print(band_10)
# print(band_20)

## 7. Build Median Composites & Export

For each year and each multi-month period:

1. Filter `COPERNICUS/S2_SR_HARMONIZED` by year, AOI, month range, and a maximum
   `CLOUDY_PIXEL_PERCENTAGE` of 20%.
2. Cloud-mask every image in the filtered collection.
3. Record per-image metadata (scene ID + cloud cover %) for later reference.
4. Reduce the collection to a **median composite**, clipped to the AOI.
5. Export each band as a separate GeoTIFF (10 m bands at 10 m scale, 20 m bands at 20 m scale),
   into a folder structure: `~/Download/S2_<year>/<year>_<start-month>-<end-month>/`.

All exports use `EPSG:32633` (UTM zone 33N) — **change this to match the UTM zone of your AOI.**

> ⚠️ This cell can take a long time to run and will make many Earth Engine export calls.
> Google Colab users: outputs are saved to the Colab VM's local disk (`~/Download`), which is
> temporary — download or copy the results to Google Drive before the session ends.


In [ ]:
base_dir = os.path.join(os.path.expanduser('~'), 'Download')

# Define the multi-month periods to composite within each year (start_month, end_month)
# months_intervals = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12)]  # bi-monthly example
months_intervals = [(1, 4), (4, 7), (7, 10), (10, 12)]  # quarterly-ish (default)

metadata_list = []

for year in years:

    year_dir = os.path.join(base_dir, f'S2_{year}')
    os.makedirs(year_dir, exist_ok=True)

    for start_month, end_month in months_intervals:

        # Filter the Sentinel-2 collection for this year/period/AOI/cloud threshold
        two_month_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .filterBounds(roi) \
            .filter(ee.Filter.calendarRange(start_month, end_month, 'month')) \
            .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 30) \
            .select(band_or10 + band_or20) \
            .map(maskS2clouds)

        num_images = two_month_collection.size().getInfo()
        if num_images == 0:
            print(f'No image found for {year} {start_month}-{end_month}, skipping ...')
            continue

        # Collect metadata for every image that went into this composite
        im_index = ee.ImageCollection(two_month_collection).aggregate_array('system:index').getInfo()
        cloudiness = ee.ImageCollection(two_month_collection).aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo()

        for idx, cloud in zip(im_index, cloudiness):
            metadata_list.append({
                'Year': year,
                'Period': f"{start_month:02d}-{end_month:02d}",
                'Image_Index': idx,
                'Cloud_Cover': cloud
            })

        # Reduce the filtered collection to a single median composite, clipped to the AOI
        two_month_image = two_month_collection.reduce(ee.Reducer.median()).clip(roi)

        period_label = f"{year}_{start_month:02d}-{end_month:02d}"
        period_dir = os.path.join(year_dir, period_label)
        os.makedirs(period_dir, exist_ok=True)

        # Print a quick summary for this period
        print(f'Year: {year}, Period: {start_month}-{end_month}, Number of images: {num_images}')
        print('Image indices:', im_index)
        print('Cloud cover:', cloudiness)

        # Export each 10 m band as its own GeoTIFF
        for band_name in band_10:
            filename = f"{band_name}_img_{period_label}.tif"
            img = two_month_image.select(band_name)
            filepath = os.path.join(period_dir, filename)
            geemap.ee_export_image(img, filename=filepath, scale=10, crs='EPSG:32633', region=roi, file_per_band=False)# Define your crs 

        # Export each 20 m band as its own GeoTIFF
        for band_name in band_20:
            filename = f"{band_name}_img_{period_label}.tif"
            img = two_month_image.select(band_name)
            filepath = os.path.join(period_dir, filename)
            geemap.ee_export_image(img, filename=filepath, scale=20, crs='EPSG:32633', region=roi, file_per_band=False)# Define your crs 

        print(f"Export completed for {period_label} in folder {base_dir}")

# Save all collected metadata (one row per source image) to a CSV file
metadata_df = pd.DataFrame(metadata_list)
metadata_csv_path = os.path.join(base_dir, 'Sentinel_metadata.csv')
metadata_df.to_csv(metadata_csv_path, index=False)
print(f"Metadata saved to {metadata_csv_path}")